# Casino Conversation Signal Detection (Phase A)

This notebook implements signal detection from multi-turn casino host–guest conversations: imports, data loading, rule-based and LLM-based detectors, and evaluation.

In [4]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 20.9 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
# --- Imports & config ---
import os
import json
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
from collections import defaultdict
from sklearn import metrics

# Load secrets from .env (project root; try cwd and parent for conv_classifiation)
load_dotenv()
load_dotenv(Path.cwd().parent / ".env")

# OpenRouter client (OpenAI-compatible API)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"  # capable chat-completion model

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
)

## Data loading

Load `casino_signal_dataset_realistic_300.jsonl` and convert to a DataFrame with `conversation` (list of message strings) and `signals` (list of signal dicts).

In [10]:
from pathlib import Path

DATA_PATH = Path("casino_signal_dataset_realistic_300.jsonl")


def load_casino_signals(path: Path) -> pd.DataFrame:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append(
                {
                    "conversation": obj.get("conversation", []),
                    "signals": obj.get("signals", []),
                }
            )
    return pd.DataFrame.from_records(records)


df = load_casino_signals(DATA_PATH)
len(df), df.head()

(300,
                                         conversation  \
 0  [Hi, I'm thinking about visiting in March., We...   
 1  [We're considering a stay soon., I'd like to b...   
 2  [We're considering a stay soon., It's actually...   
 3  [I'm planning a trip in August., Last time I h...   
 4  [We might come by next month., I'm trying to p...   
 
                                              signals  
 0  [{'type': 'value', 'value': 'spends heavily on...  
 1  [{'type': 'intent', 'value': 'wants to book a ...  
 2     [{'type': 'life_event', 'value': 'promotion'}]  
 3  [{'type': 'sentiment', 'value': 'loved the cas...  
 4  [{'type': 'intent', 'value': 'planning a trip'...  )

## Exploratory analysis

Inspect a few examples and summarize label distribution by signal `type` to understand the dataset.

In [11]:
# Peek at a few random conversations
sample_df = df.sample(5, random_state=0)
for i, row in sample_df.iterrows():
    print(f"--- Example {i} ---")
    for turn_idx, msg in enumerate(row["conversation"], start=1):
        print(f"[{turn_idx}] {msg}")
    print("Signals:")
    for sig in row["signals"]:
        print(f"  - type={sig.get('type')} | value={sig.get('value')}")
    print()


# Flatten all signals to analyze label distribution
all_signal_types = []
for sigs in df["signals"]:
    for sig in sigs:
        sig_type = (sig.get("type") or "").strip().lower()
        if sig_type:
            all_signal_types.append(sig_type)

value_counts = pd.Series(all_signal_types).value_counts().sort_index()
print("Signal type counts:\n", value_counts)
print("\nSignal type proportions:\n", (value_counts / value_counts.sum()).round(3))

--- Example 208 ---
[1] We might come by next month.
Signals:

--- Example 188 ---
[1] We're considering a stay soon.
Signals:

--- Example 12 ---
[1] We're considering a stay soon.
[2] I really enjoyed the casino last visit.
[3] We normally stay in suites.
[4] I'd like to book something.
Signals:
  - type=sentiment | value=service was not what they expected
  - type=value | value=usually stays in suites
  - type=intent | value=wants to book a room

--- Example 221 ---
[1] We might come by next month.
Signals:

--- Example 239 ---
[1] I wanted to ask about booking options.
[2] I'd like to book something.
[3] Last time I had a great experience.
Signals:
  - type=intent | value=wants to reserve the steakhouse
  - type=sentiment | value=frustrated about a previous issue

Signal type counts:
 competitive    89
intent         99
life_event     88
sentiment      86
value          91
Name: count, dtype: int64

Signal type proportions:
 competitive    0.196
intent         0.219
life_event     

## Signal ontology and ground truth representation

Define a canonical set of signal `type` labels and convert each conversation's `signals` list into a set of normalized `(type, value)` tuples for evaluation.

In [12]:
CANONICAL_SIGNAL_TYPES = {"intent", "value", "sentiment", "life_event", "competitive"}


def normalize_signal_type(raw: str) -> str | None:
    if not raw:
        return None
    t = raw.strip().lower().replace("-", "_").replace(" ", "_")
    # Simple aliasing if needed
    if t == "lifeevent":
        t = "life_event"
    if t in CANONICAL_SIGNAL_TYPES:
        return t
    return None


def normalize_signal_value(raw: str) -> str | None:
    if not raw:
        return None
    v = raw.strip().lower()
    return v or None


def signals_to_set(signals: list[dict]) -> set[tuple[str, str]]:
    result: set[tuple[str, str]] = set()
    for sig in signals:
        t = normalize_signal_type(sig.get("type"))
        v = normalize_signal_value(sig.get("value"))
        if t and v:
            result.add((t, v))
    return result


# Add ground-truth signal sets as a convenience column for later evaluation
df["gt_signal_set"] = df["signals"].apply(signals_to_set)

# Quick sanity check
df[["signals", "gt_signal_set"]].head()

,signals,gt_signal_set
0,"[{'type': 'value', 'value': 'spends heavily on...","{(value, spends heavily on dining), (competiti..."
1,"[{'type': 'intent', 'value': 'wants to book a ...","{(intent, wants to book a room), (life_event, ..."
2,"[{'type': 'life_event', 'value': 'promotion'}]","{(life_event, promotion)}"
3,"[{'type': 'sentiment', 'value': 'loved the cas...","{(sentiment, loved the casino atmosphere), (in..."
4,"[{'type': 'intent', 'value': 'planning a trip'...","{(intent, planning a trip), (life_event, honey..."


## Baseline rule-based signal detector

Implement a simple keyword/phrase-based detector `detect_signals_rules(messages: list[str]) -> list[dict]` that emits heuristic signals with confidence scores.

In [13]:
from typing import List, Dict, Any


def detect_signals_rules(messages: List[str]) -> List[Dict[str, Any]]:
    """Simple keyword/phrase-based detector for casino conversation signals.

    Returns a list of dicts with keys: type, value, confidence, rationale.
    """
    full_text = " ".join(messages).lower()

    signals: List[Dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()

    def add_signal(sig_type: str, value: str, confidence: float, rationale: str) -> None:
        sig_type_norm = normalize_signal_type(sig_type)
        value_norm = normalize_signal_value(value)
        if not sig_type_norm or not value_norm:
            return
        key = (sig_type_norm, value_norm)
        if key in seen:
            return
        seen.add(key)
        signals.append(
            {
                "type": sig_type_norm,
                "value": value_norm,
                "confidence": float(max(0.0, min(1.0, confidence))),
                "rationale": rationale,
            }
        )

    # --- Intent signals ---
    intent_booking_phrases = [
        "book a room",
        "book something",
        "make a reservation",
        "reserve a room",
        "reservation",
        "help me reserve",
        "ask about booking options",
    ]
    if any(p in full_text for p in intent_booking_phrases):
        add_signal(
            "intent",
            "wants to book a room",
            0.85,
            "matched booking-related phrase",
        )

    intent_planning_phrases = [
        "planning a trip",
        "plan my stay",
        "planning to visit",
        "might come by next month",
        "considering a stay soon",
    ]
    if any(p in full_text for p in intent_planning_phrases):
        add_signal(
            "intent",
            "planning a trip",
            0.8,
            "matched planning-related phrase",
        )

    # --- Value signals ---
    if "stay in suites" in full_text or "stay in a suite" in full_text or "normally stay in suites" in full_text:
        add_signal(
            "value",
            "usually stays in suites",
            0.8,
            "mentions usually staying in suites",
        )

    if "budget isn't a concern" in full_text or "spend a lot" in full_text or "spends heavily" in full_text:
        add_signal(
            "value",
            "spends heavily on property",
            0.75,
            "mentions high spend / budget not a concern",
        )

    # --- Life event signals ---
    if "honeymoon" in full_text:
        add_signal(
            "life_event",
            "honeymoon",
            0.9,
            "explicit mention of honeymoon",
        )
    if "anniversary" in full_text:
        add_signal(
            "life_event",
            "anniversary",
            0.9,
            "explicit mention of anniversary",
        )
    if "birthday" in full_text:
        add_signal(
            "life_event",
            "birthday",
            0.9,
            "explicit mention of birthday",
        )
    if "promotion" in full_text:
        add_signal(
            "life_event",
            "promotion",
            0.9,
            "explicit mention of promotion",
        )

    # --- Sentiment signals ---
    if "great experience" in full_text or "really enjoyed" in full_text or "loved the casino" in full_text:
        add_signal(
            "sentiment",
            "positive prior experience",
            0.8,
            "mentions positive previous stay",
        )

    if "frustrated" in full_text or "bad experience" in full_text or "not what i expected" in full_text:
        add_signal(
            "sentiment",
            "negative prior experience",
            0.8,
            "mentions negative previous stay",
        )

    # --- Competitive signals ---
    competitor_names = ["wynn", "bellagio", "cosmopolitan"]
    for comp in competitor_names:
        if comp in full_text:
            add_signal(
                "competitive",
                comp,
                0.9,
                "mentions competitor property by name",
            )

    return signals


# Apply rule-based detector over the dataset
df["rules_signals"] = df["conversation"].apply(detect_signals_rules)
df["rules_signal_set"] = df["rules_signals"].apply(signals_to_set)

# Inspect a few examples to compare ground truth vs rule-based predictions
df[["conversation", "gt_signal_set", "rules_signal_set"]].head(5)

,conversation,gt_signal_set,rules_signal_set
0,"[Hi, I'm thinking about visiting in March., We...","{(value, spends heavily on dining), (competiti...","{(value, usually stays in suites), (competitiv..."
1,"[We're considering a stay soon., I'd like to b...","{(intent, wants to book a room), (life_event, ...","{(intent, planning a trip), (intent, wants to ..."
2,"[We're considering a stay soon., It's actually...","{(life_event, promotion)}","{(intent, planning a trip), (life_event, promo..."
3,"[I'm planning a trip in August., Last time I h...","{(sentiment, loved the casino atmosphere), (in...","{(intent, planning a trip), (intent, wants to ..."
4,"[We might come by next month., I'm trying to p...","{(intent, planning a trip), (life_event, honey...","{(intent, planning a trip), (life_event, honey..."


## LLM-based signal detector via OpenRouter

Use a chat-completion model to extract signals from each conversation. System prompt defines the five signal categories and required JSON output; responses are parsed and normalized (canonical types, confidence in [0,1]).

In [14]:
SYSTEM_PROMPT = """You are analyzing a multi-turn conversation between a casino host and a guest. Your task is to detect hospitality-relevant "signals" that the guest expresses (explicitly or strongly implied). Only include signals that are clearly present in the conversation.

Signal categories (use exactly these "type" values):
- intent: Guest's stated or implied intention (e.g. planning a trip, wants to book a room, wants to reserve dining).
- value: What the guest values or prefers (e.g. usually stays in suites, spends heavily on dining, budget isn't a concern).
- sentiment: Positive or negative feeling about a prior stay or the property (e.g. loved the casino atmosphere, frustrated about a previous issue).
- life_event: Special occasion or milestone (e.g. honeymoon, anniversary, birthday, promotion).
- competitive: Mention of or preference for a competitor property (e.g. Wynn, Bellagio, Cosmopolitan).

Respond with a single JSON object of this form (no other text):
{"signals": [{"type": "<one of the five types>", "value": "<short natural-language description>", "confidence": <number in (0, 1]>, "rationale": "<brief reason>"}, ...]}

If there are no clear signals, use: {"signals": []}."""


def format_conversation(messages: list[str]) -> str:
    """Turn a list of message strings into a readable transcript."""
    return "\n".join(f"Turn {i+1}: {msg}" for i, msg in enumerate(messages))


def call_llm(conversation: list[str]) -> dict:
    """Call OpenRouter chat-completions and return parsed JSON. Retries once on parse error."""
    transcript = format_conversation(conversation)
    user_content = f"Conversation:\n\n{transcript}\n\nExtract all relevant signals and respond with the JSON object only."

    for attempt in range(2):
        try:
            resp = client.chat.completions.create(
                model=OPENROUTER_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_content},
                ],
                response_format={"type": "json_object"},
            )
            content = (resp.choices[0].message.content or "").strip()
            # Strip optional markdown code fence
            if content.startswith("```"):
                lines = content.split("\n")
                if lines[0].startswith("```"):
                    lines = lines[1:]
                if lines and lines[-1].strip() == "```":
                    lines = lines[:-1]
                content = "\n".join(lines)
            return json.loads(content)
        except (json.JSONDecodeError, KeyError, IndexError) as e:
            if attempt == 0:
                user_content = user_content + "\n\n[Previous response was invalid. Reply with only a valid JSON object, no markdown or extra text.]"
                continue
            raise RuntimeError(f"LLM response parse failed after retry: {e}") from e

    raise RuntimeError("call_llm failed unexpectedly")


def detect_signals_llm(messages: list[str]) -> list[dict]:
    """Run LLM-based signal detection and return normalized list of signal dicts."""
    raw = call_llm(messages)
    signals_in = raw.get("signals") or []
    out = []
    for s in signals_in:
        if not isinstance(s, dict):
            continue
        t = normalize_signal_type(s.get("type"))
        if t not in CANONICAL_SIGNAL_TYPES:
            continue
        v = (s.get("value") or "").strip()
        if not v:
            continue
        v_norm = normalize_signal_value(v) or v.lower().strip()
        try:
            conf = float(s.get("confidence", 0.5))
        except (TypeError, ValueError):
            conf = 0.5
        conf = max(0.0, min(1.0, conf))
        out.append({
            "type": t,
            "value": v_norm,
            "confidence": conf,
            "rationale": (s.get("rationale") or "").strip() or "LLM extraction",
        })
    return out


# Optional: run on first 3 rows to verify (comment out or expand to run on full df)
# df["llm_signals"] = df["conversation"].apply(detect_signals_llm)
# df["llm_signal_set"] = df["llm_signals"].apply(signals_to_set)
_sample = df["conversation"].iloc[:3]
for i, msgs in enumerate(_sample):
    pred = detect_signals_llm(msgs)
    print(f"--- Conversation {i} ---")
    for p in pred:
        print(f"  {p}")
    print()

--- Conversation 0 ---
  {'type': 'intent', 'value': 'planning a visit in march', 'confidence': 0.9, 'rationale': 'The guest explicitly states they are thinking about visiting.'}
  {'type': 'value', 'value': 'usually stays in suites', 'confidence': 0.8, 'rationale': 'The guest mentions their usual accommodation preference.'}
  {'type': 'competitive', 'value': 'often goes to wynn instead', 'confidence': 0.85, 'rationale': 'The guest indicates a preference for another casino property.'}

--- Conversation 1 ---
  {'type': 'intent', 'value': 'guest wants to book a stay soon', 'confidence': 0.9, 'rationale': 'The guest explicitly states a desire to book something.'}
  {'type': 'competitive', 'value': 'guest often chooses wynn instead', 'confidence': 0.8, 'rationale': 'The guest mentions a preference for Wynn, indicating a competitive comparison.'}
  {'type': 'life_event', 'value': "it's a special trip for a birthday", 'confidence': 0.95, 'rationale': 'Guest explicitly states this stay is fo